In [1]:
# 필요한 라이브러리들을 임포트합니다.
from sklearn.linear_model import LogisticRegression # 로지스틱 회귀 모델
from sklearn.ensemble import RandomForestClassifier # 랜덤 포레스트 분류기
from sklearn.model_selection import train_test_split, GridSearchCV # 훈련/테스트 데이터 분할 및 그리드 서치를 위한 모듈
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif # 특성 선택을 위한 모듈 (최고 K개 선택, 분산 임계값, ANOVA F-값)
from sklearn.tree import DecisionTreeClassifier # 결정 트리 분류기
from sklearn.metrics import roc_auc_score, fbeta_score, make_scorer # ROC AUC 점수, F-베타 점수, 커스텀 스코어러 생성
from xgboost import XGBClassifier # XGBoost 분류기
import shap # SHAP(SHapley Additive exPlanations) 라이브러리 (모델 예측 설명)
import matplotlib.pyplot as plt # 데이터 시각화를 위한 라이브러리

import pandas as pd # 데이터 조작 및 분석을 위한 라이브러리
import numpy as np # 수치 계산을 위한 라이브러리
import datetime as dt # 날짜 및 시간 처리를 위한 라이브러리
import json # JSON 데이터 처리를 위한 라이브러리

In [2]:
# 경고 메시지 처리를 위한 모듈
import warnings 

# 'use_label_encoder' 경고만 무시합니다.
warnings.filterwarnings("ignore")

#### prepare "data/initial_dataset.p"

In [3]:
# .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3

# # 파일 경로 지정
# file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.xlsx'

# # 엑셀 파일을 DataFrame으로 읽어오기
# # 기본적으로 첫 번째 시트를 읽어옵니다.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [4]:
# .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress

# # 파일 경로 지정
# file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.xlsx'

# # 엑셀 파일을 DataFrame으로 읽어오기
# # 기본적으로 첫 번째 시트를 읽어옵니다.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)

In [5]:
# read *.p
pickle_file_path_1 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'
data_row_1 = pd.read_pickle(pickle_file_path_1)
pickle_file_path_2 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'
data_row_2 = pd.read_pickle(pickle_file_path_2)

In [6]:
# 필요한 컬럼만 keep

# 파일 경로 지정
file_path = 'data/cols_to_keep.csv'

# CSV 파일을 DataFrame으로 읽어오기
cols_to_keep_df = pd.read_csv(file_path)

cols_to_keep = cols_to_keep_df.iloc[:, 0].tolist()

data_row_1 = data_row_1[cols_to_keep]

In [7]:
# data_row <= data_row1 data_row2

# data_row_2에서 조인할 컬럼만 선택
columns_to_join = ['DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP',
                  #  'wafer_id', 
                #    'SensorOffsetHot-RoomAfterBake', 
                #    'SensorOffsetHot-ColdAfterBake', 
                   'BG pass/fail']

# 선택한 컬럼으로 data_row_2의 부분집합 DataFrame 생성
data_row_2_subset = data_row_2[columns_to_join]

# data_row_1에 data_row_2의 선택된 컬럼들을 조인 키 'DevID'로 병합
data_row = pd.merge(data_row_1, data_row_2_subset, on='DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP', how='left')

# # 결과 DataFrame 확인
# print(merged_df.head())

In [8]:
initial_dataset = data_row.copy() # 원본 데이터셋 복사
# processed_dataset = initial_dataset.copy() # 원본 데이터셋 복사

In [9]:
# prepare for target

initial_dataset['Pass/Fail_pass'] = ((initial_dataset['soft_bin of FT1'] == 1) &
                   (initial_dataset['soft_bin of FT2'] == 1) &
                   (initial_dataset['soft_bin'] == 1)).astype(int)

In [10]:
# prepare for base model

initial_dataset['band gap dpat'] = initial_dataset['BG pass/fail'].apply(lambda x: 'bandGapFail' if x == 'impossible wafer' else 'ok for band gap')

# 컬럼 이름 변경 딕셔너리 생성
new_column_names = {
    'wafer_id': 'WAFER_NO',
    'DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP': 'DevID'
}

# .rename() 메서드를 사용하여 컬럼 이름 변경 (inplace=True로 원본 데이터프레임에 바로 적용)
initial_dataset.rename(columns=new_column_names, inplace=True)

# 변경된 컬럼 이름 확인
# print(initial_dataset.columns)

In [11]:
# initial_dataset

In [12]:
# 제거할 컬럼 리스트 정의
columns_to_drop = [
    'soft_bin of FT1',
    'soft_bin of FT2',
    'soft_bin',
    'BG pass/fail'
]

# 컬럼 drop (원본 DataFrame을 변경하려면 inplace=True 사용)
# 또는 새로운 DataFrame을 만들려면 processed_dataset = processed_dataset.drop(...) 사용
initial_dataset.drop(columns=columns_to_drop, inplace=True)

In [13]:
# Rename the columns
initial_dataset.rename(columns={'x_pos': 'X', 'y_pos': 'Y'}, inplace=True)

In [14]:
# Radius 컬럼 계산
# np.sqrt() 함수는 각 요소의 제곱근을 계산합니다.
initial_dataset['Radius'] = np.sqrt(initial_dataset['X']**2 + initial_dataset['Y']**2)

In [15]:
initial_dataset.to_pickle("data/initial_dataset.p")

#### scnarios

In [16]:
# import vars and function

from algos.algos import *
from config.config import *

In [17]:
import copy

##### preprocess_dataset

In [18]:
# def preprocess_dataset(initial_dataset: pd.DataFrame):
    # return processed_dataset # 전처리된 데이터셋 반환

preprocessed_dataset = preprocess_dataset(initial_dataset)

##### create_train_and_test_data
# def create_train_test_data(
#     preprocessed_dataset: pd.DataFrame,
#     split_parameter: dict = None
# ):
#     return train_data, test_data, split_parameter_info



     데이터셋 전처리 중...
     전처리 완료!



##### create_train_and_test_data

In [19]:
# def create_train_test_data(
#     preprocessed_dataset: pd.DataFrame,
#     split_parameter: dict = None
# ):
#     return train_data, test_data, split_parameter_info

In [20]:
split_parameter = copy.deepcopy(split_parameter_default)

In [21]:
split_parameter_default

{'test_size': 0.2,
 'random_state': 42,
 'apply_filter_split': False,
 'var_threshold_split': 0.0,
 'corr_threshold_split': 0.98,
 'sampling_method': None,
 'sampling_ratio': None,
 'apply_feature_generation': False,
 'sum_features': False,
 'diff_features': False,
 'poly_features': False,
 'poly_degree': 2,
 'apply_filter_gen': False,
 'var_threshold_gen': 0.0,
 'corr_threshold_gen': 0.1}

In [22]:
params = [
    [None, None],
    ["ROS", 1.0],
    ["SMOTE", 1.0],
    ["ADASYN", 1.0]

    # ["ROS", 1.0],
    # ["ROS", 1.5],
    # ["ROS", 2.0],
    # ["SMOTE", 1.0],
    # ["SMOTE", 1.5],
    # ["SMOTE", 2.0],
    # ["ADASYN", 1.0],
    # ["ADASYN", 1.5],
    # ["ADASYN", 2.0]
]

In [23]:
# split_parameter["sampling_method"] = params[0]
# split_parameter["sampling_ratio"] = 1.2

split_parameters = {}
for i, param in enumerate(params):
    split_parameter = copy.deepcopy(split_parameter_default)
    split_parameter["sampling_method"] = param[0]
    split_parameter["sampling_ratio"] = param[1]
    split_parameters[i] = split_parameter

# print(split_parameters)

In [24]:
# for key, value in split_parameters.items():
#     print(f"Key: {key}")
#     print(f"Value: {value}")
#     print("-" * 20)

In [25]:
# train_data, test_data, split_parameter_info = create_train_test_data(preprocessed_dataset, split_parameter)

In [26]:
feature_selector_params_sfm = copy.deepcopy(feature_selector_params_sfm_default)
feature_selector_params_sfm["params"]["estimator"]["params"]["n_estimators"] = 250 # max = len(train_data.columns) - 1
feature_selector_params_sfm["params"]["threshold"] = "1*median"
# feature_selection_info_sfm = select_feature(train_data, feature_selector_params_sfm)

##### 샘플링 방법 > 성능체크 : 여러 샘플링, sfm 변수선택, rf_cv 모델 파이프라인 활용

In [27]:
# feature select test : pipeline
## 선택한 피처셋 feature_selection_info['final_features'] 리스트를 모델링 파이프라인에 적용, 모델링 결과를 리턴

def pl_smpl_test(
        preprocessed_dataset,
        split_parameter,
        feature_selector_params,
        train_parameters,
        ):
    
    train_data, test_data, split_parameter_info = create_train_test_data(preprocessed_dataset, split_parameter)
    feature_selection_info = select_feature(train_data, feature_selector_params)
    trained_model, feature_importance, train_parameters_info = train_model_rf_cv(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        split_parameter_info, \
        feature_selection_info, \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results

In [ ]:
# 지정한 샘플링방법에 대한 모델적용 및 결과 정리
# sampling performance test - submit and summary result

usr_pl_smpl_test_result_ftpn_df = pd.DataFrame()
# usr_pl_smpl_test_result_features_values_dfs = pd.DataFrame(
#     list(train_data.columns), 
#     columns=['feature_name']
# )

for idx, split_parameter in split_parameters.items():
    split_parameter_info, \
    feature_selection_info, \
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_smpl_test(
            preprocessed_dataset,
            split_parameter,
            feature_selector_params_sfm,
            train_parameters_list_default["rf_cv"],
            )

    dict_ftpn = metrics["dict_ftpn"]

    if "class_distribution_after_sampling" in split_parameter_info :
        class_distribution = split_parameter_info["class_distribution_after_sampling"]
    else :
        class_distribution = split_parameter_info["class_distribution_before_sampling"]
        
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'sampling_method_used' : split_parameter_info["sampling_method_used"],
        'sampling_ratio_used' : split_parameter_info["sampling_ratio_used"],
        'class_distribution_before_sampling' : split_parameter_info["class_distribution_before_sampling"],
        'class_distribution_after_sampling' : class_distribution,
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'filter_methods' : feature_selection_info["filter_methods"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'feature_selection_info_json_path' : feature_selection_info["feature_selection_info_json_path"]
    }
    
    usr_pl_smpl_test_result_ftpn_df = pd.concat([usr_pl_smpl_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)

    if feature_selection_info["feature_selector_name"] == 'FeatureFilter':
        if feature_selection_info["filter_methods"]["apply_variance_filter"] == True:
            value_type = "variance"
        elif feature_selection_info["filter_methods"]["apply_target_linear_corr_filter"] == True:
            value_type = "target_linear_correlation"
        elif feature_selection_info["filter_methods"]["apply_target_xicor_filter"] == True:
            value_type = "target_xicor_correlation"
        features_values = feature_selection_info["selection_details"][value_type]["features_values_checked"]
    else : 
        value_type = "importances"
        features_values = feature_selection_info["selection_details"][value_type]

    # features_values_df = pd.DataFrame(
    #     list(features_values.items()), 
    #     columns=['feature_name', feature_selection_info["feature_selector_name"]+"_"+value_type]
    # )

    # usr_pl_smpl_test_result_features_values_dfs = pd.merge(
    #     usr_pl_smpl_test_result_features_values_dfs,
    #     features_values_df,
    #     how='outer',
    #     left_on='feature_name',
    #     right_on='feature_name'
    #     )



##############################################################################################################################
# 3) Create Train/Test Split (훈련/테스트 데이터 분할) 
##############################################################################################################################

     훈련 및 테스트 데이터셋 생성 중...
     - 분할 전 필터링 미적용.
     - Feature Generation 미적용.

     - 분할 전 훈련 데이터 클래스 분포: {0.0: 3546, 1.0: 71}
     - 샘플링 미적용

--- 피처 선택기: SFM ---
--- SFM 선택기 완료 ---
남은 피처 수: 825

피처 선택 결과가 'data/result/jsons\feature_selection_info_250901_152536_0ca2f459.json' 파일에 저장되었습니다.

- 최종 피처 수: 825
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.2297

      Forecasting the test dataset...
      Forecasting done!
Best threshold for F2 score: 0.7980 with F2 sco

In [ ]:
# 샘플링방법 모델성능요약

usr_pl_smpl_test_result_ftpn_df

In [ ]:
# 피처선택셋 모델성능요약 + 사용자 피처선택 임계값 적용 컬럼(참고용)

# usr_pl_smpl_test_result_features_values_dfs